# PINN para Movimiento Parabólico

Este cuaderno implementa una **Physics-Informed Neural Network (PINN)** para aproximar la trayectoria de un proyectil en 2D.

## Modelo físico
Sin rozamiento del aire:

\[
\frac{d^2x}{dt^2}=0,\qquad \frac{d^2y}{dt^2}=-g
\]

con condiciones iniciales:

\[
x(0)=x_0,\ y(0)=y_0,\ \dot{x}(0)=v_0\cos\theta,\ \dot{y}(0)=v_0\sin\theta
\]

La PINN aprende funciones \(x(t)\) y \(y(t)\) minimizando simultáneamente:

- Residuales de la ecuación diferencial (física)
- Condiciones iniciales
- Ajuste a pocos datos observados (opcional)

Al final se compara contra la solución analítica.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Parámetros físicos del proyectil
g = 9.81
v0 = 20.0
theta_deg = 50.0
theta = np.deg2rad(theta_deg)
x0, y0 = 0.0, 0.0

vx0 = v0 * np.cos(theta)
vy0 = v0 * np.sin(theta)

# Tiempo de vuelo aproximado para y0 = 0
T = 2 * vy0 / g
print(f'Tiempo de vuelo aproximado: {T:.3f} s')

# Solución analítica de referencia
def trayectoria_analitica(t):
    x = x0 + vx0 * t
    y = y0 + vy0 * t - 0.5 * g * t**2
    return x, y

# Puntos de observación (pocos datos)
N_data = 25
t_data_np = np.linspace(0, T, N_data).reshape(-1, 1)
x_data_np, y_data_np = trayectoria_analitica(t_data_np)

# Tensores de datos
t_data = torch.tensor(t_data_np, dtype=torch.float32, device=device)
x_data = torch.tensor(x_data_np, dtype=torch.float32, device=device)
y_data = torch.tensor(y_data_np, dtype=torch.float32, device=device)

# Puntos de colación para imponer física
N_f = 200
t_f = torch.linspace(0, T, N_f, device=device).view(-1, 1)
t_f.requires_grad_(True)

# Tensor para t=0 (condiciones iniciales)
t0 = torch.zeros((1, 1), dtype=torch.float32, device=device, requires_grad=True)

In [ ]:
class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        mods = []
        for i in range(len(layers) - 2):
            mods.append(nn.Linear(layers[i], layers[i + 1]))
            mods.append(nn.Tanh())
        mods.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*mods)

    def forward(self, t):
        return self.net(t)  # salida: [x(t), y(t)]


layers = [1, 64, 64, 64, 2]
model = PINN(layers).to(device)


def grad(outputs, inputs):
    return torch.autograd.grad(
        outputs,
        inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=True,
        retain_graph=True
    )[0]


def pinn_losses(model, t_f, t_data, x_data, y_data, t0):
    # Física en puntos de colación
    pred_f = model(t_f)
    x_f = pred_f[:, 0:1]
    y_f = pred_f[:, 1:2]

    x_t = grad(x_f, t_f)
    y_t = grad(y_f, t_f)
    x_tt = grad(x_t, t_f)
    y_tt = grad(y_t, t_f)

    res_x = x_tt
    res_y = y_tt + g
    loss_phys = (res_x**2).mean() + (res_y**2).mean()

    # Condiciones iniciales en t = 0
    pred0 = model(t0)
    x0_pred = pred0[:, 0:1]
    y0_pred = pred0[:, 1:2]

    x_t0 = grad(x0_pred, t0)
    y_t0 = grad(y0_pred, t0)

    loss_ic = (
        (x0_pred - x0) ** 2 +
        (y0_pred - y0) ** 2 +
        (x_t0 - vx0) ** 2 +
        (y_t0 - vy0) ** 2
    ).mean()

    # Ajuste a datos observados
    pred_data = model(t_data)
    x_data_pred = pred_data[:, 0:1]
    y_data_pred = pred_data[:, 1:2]
    loss_data = ((x_data_pred - x_data) ** 2).mean() + ((y_data_pred - y_data) ** 2).mean()

    # Ponderación de términos
    w_phys, w_ic, w_data = 1.0, 10.0, 1.0
    total = w_phys * loss_phys + w_ic * loss_ic + w_data * loss_data

    return total, loss_phys, loss_ic, loss_data

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5000
history = {'total': [], 'phys': [], 'ic': [], 'data': []}

for ep in range(1, epochs + 1):
    optimizer.zero_grad()

    loss, l_phys, l_ic, l_data = pinn_losses(model, t_f, t_data, x_data, y_data, t0)
    loss.backward()
    optimizer.step()

    history['total'].append(loss.item())
    history['phys'].append(l_phys.item())
    history['ic'].append(l_ic.item())
    history['data'].append(l_data.item())

    if ep % 500 == 0:
        print(
            f"Epoch {ep:5d} | Total: {loss.item():.3e} | "
            f"Phys: {l_phys.item():.3e} | IC: {l_ic.item():.3e} | Data: {l_data.item():.3e}"
        )

In [ ]:
# Curvas de pérdida
plt.figure(figsize=(8, 4))
plt.semilogy(history['total'], label='Total')
plt.semilogy(history['phys'], label='Física')
plt.semilogy(history['ic'], label='IC')
plt.semilogy(history['data'], label='Datos')
plt.xlabel('Época')
plt.ylabel('Loss (escala log)')
plt.title('Evolución de pérdidas de la PINN')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Evaluación sobre una malla fina de tiempo
model.eval()
with torch.no_grad():
    t_test_np = np.linspace(0, T, 300).reshape(-1, 1)
    t_test = torch.tensor(t_test_np, dtype=torch.float32, device=device)

    pred = model(t_test).cpu().numpy()
    x_pred = pred[:, 0]
    y_pred = pred[:, 1]

x_true, y_true = trayectoria_analitica(t_test_np)
x_true = x_true.flatten()
y_true = y_true.flatten()

# Error medio absoluto
mae_x = np.mean(np.abs(x_pred - x_true))
mae_y = np.mean(np.abs(y_pred - y_true))
print(f'MAE en x(t): {mae_x:.4e}')
print(f'MAE en y(t): {mae_y:.4e}')

In [ ]:
# Comparación de trayectoria en el plano x-y
plt.figure(figsize=(7, 5))
plt.plot(x_true, y_true, 'k-', linewidth=2, label='Analítica')
plt.plot(x_pred, y_pred, 'r--', linewidth=2, label='PINN')
plt.scatter(x_data_np, y_data_np, c='royalblue', s=25, alpha=0.7, label='Datos')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Movimiento parabólico: Analítica vs PINN')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Comparación temporal
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t_test_np, x_true, 'k-', label='x analítica')
ax[0].plot(t_test_np, x_pred, 'r--', label='x PINN')
ax[0].set_xlabel('t (s)')
ax[0].set_ylabel('x (m)')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(t_test_np, y_true, 'k-', label='y analítica')
ax[1].plot(t_test_np, y_pred, 'r--', label='y PINN')
ax[1].set_xlabel('t (s)')
ax[1].set_ylabel('y (m)')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.suptitle('Comparación temporal de estados')
plt.tight_layout()
plt.show()